In [4]:
import pandas as pd

df = pd.read_csv('superstore_dataset2011-2015.csv', encoding='latin1')
print(df.shape)
print(df.head())

#What this does:

#pandas is the library used for data manipulation in Python
#read_csv loads your CSV file into a dataframe (think of it like a table)
#encoding='latin1' handles special characters in the file
#df.shape shows you rows and columns count
#df.head() shows first 5 rows to confirm it loaded correctly

(51290, 24)
   Row ID         Order ID Order Date Ship Date       Ship Mode Customer ID  \
0   42433     AG-2011-2040   1/1/2011  6/1/2011  Standard Class    TB-11280   
1   22253    IN-2011-47883   1/1/2011  8/1/2011  Standard Class    JH-15985   
2   48883     HU-2011-1220   1/1/2011  5/1/2011    Second Class      AT-735   
3   11731  IT-2011-3647632   1/1/2011  5/1/2011    Second Class    EM-14140   
4   22255    IN-2011-47883   1/1/2011  8/1/2011  Standard Class    JH-15985   

     Customer Name      Segment         City            State  ...  \
0  Toby Braunhardt     Consumer  Constantine      Constantine  ...   
1      Joseph Holt     Consumer  Wagga Wagga  New South Wales  ...   
2    Annie Thurman     Consumer     Budapest         Budapest  ...   
3     Eugene Moren  Home Office    Stockholm        Stockholm  ...   
4      Joseph Holt     Consumer  Wagga Wagga  New South Wales  ...   

         Product ID         Category Sub-Category  \
0  OFF-TEN-10000025  Office Supplies   

In [5]:
df.drop(columns=['Row ID', 'Postal Code'], inplace=True)
print(df.columns.tolist())

#drop removes columns we don't need
#Row ID is just a sequence number, no analytical value
#Postal Code has 80% nulls and is useless for our analysis
#inplace=True means apply the change directly to df without creating a new variable
#df.columns.tolist() prints remaining columns so you can verify

['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'City', 'State', 'Country', 'Market', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Shipping Cost', 'Order Priority']


In [7]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed', dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='mixed', dayfirst=True)

print(df[['Order Date', 'Ship Date']].head())

#pd.to_datetime converts text dates into proper date format Python understands
#format='mixed' handles the mixed date formats in our dataset (some use / some use -)
#dayfirst=True tells Python the dates are DD/MM/YYYY not MM/DD/YYYY — this is the fix for the Power Query problem you faced
#Print to verify dates look correct

  Order Date  Ship Date
0 2011-01-01 2011-01-06
1 2011-01-01 2011-01-08
2 2011-01-01 2011-01-05
3 2011-01-01 2011-01-05
4 2011-01-01 2011-01-08


In [8]:
df['Delivery Days'] = (df['Ship Date'] - df['Order Date']).dt.days

print(df['Delivery Days'].describe())


#Subtracts Order Date from Ship Date to get the difference
#.dt.days converts that difference into a plain number (days)
#describe() shows min, max, average — you should see average around 4 days

count    51290.000000
mean         3.969370
std          1.729437
min          0.000000
25%          3.000000
50%          4.000000
75%          5.000000
max          7.000000
Name: Delivery Days, dtype: float64


In [10]:
df['Sales'] = df['Sales'].astype(float)
df['Profit'] = df['Profit'].astype(float)
df['Discount'] = df['Discount'].astype(float)
df['Shipping Cost'] = df['Shipping Cost'].astype(float)
df['Quantity'] = df['Quantity'].astype(int)

print(df.dtypes)

#astype(float) converts columns to decimal numbers so Power BI can calculate with them
#astype(int) converts Quantity to whole number since you can't have 2.5 items
#df.dtypes prints all column types so you can verify everything looks correct

Order ID                  object
Order Date        datetime64[ns]
Ship Date         datetime64[ns]
Ship Mode                 object
Customer ID               object
Customer Name             object
Segment                   object
City                      object
State                     object
Country                   object
Market                    object
Region                    object
Product ID                object
Category                  object
Sub-Category              object
Product Name              object
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
Shipping Cost            float64
Order Priority            object
Delivery Days              int64
dtype: object


In [11]:
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Month Name'] = df['Order Date'].dt.strftime('%B')

print(df[['Order Date', 'Year', 'Month', 'Month Name']].head())

#Extracts Year and Month from Order Date as separate columns
#These make time-based filtering much easier in Power BI
#strftime('%B') gives the full month name like "January" instead of just 1
#Print to verify the new columns look right

  Order Date  Year  Month Month Name
0 2011-01-01  2011      1    January
1 2011-01-01  2011      1    January
2 2011-01-01  2011      1    January
3 2011-01-01  2011      1    January
4 2011-01-01  2011      1    January


In [12]:
text_cols = ['Ship Mode', 'Customer Name', 'Segment', 'City',
             'State', 'Country', 'Market', 'Region',
             'Category', 'Sub-Category', 'Product Name', 'Order Priority']

for col in text_cols:
    df[col] = df[col].str.strip()

print("Text columns cleaned!")

#str.strip() removes any invisible leading or trailing spaces from text
#For example " Consumer " becomes "Consumer"
#This prevents duplicates in Power BI filters caused by hidden spaces
#The for loop applies it to all text columns at once

Text columns cleaned!


In [13]:
print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Null values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Date range:", df['Order Date'].min(), "to", df['Order Date'].max())
print("Delivery Days range:", df['Delivery Days'].min(), "to", df['Delivery Days'].max())

#This is your quality check before exporting
#You should see zero nulls, zero duplicates
#Delivery Days should show 0 to 7
#Date range should show 2011 to 2014


Total rows: 51290
Total columns: 26
Null values:
 Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
City              0
State             0
Country           0
Market            0
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
Quantity          0
Discount          0
Profit            0
Shipping Cost     0
Order Priority    0
Delivery Days     0
Year              0
Month             0
Month Name        0
dtype: int64
Duplicate rows: 0
Date range: 2011-01-01 00:00:00 to 2014-12-31 00:00:00
Delivery Days range: 0 to 7


In [14]:
df.to_csv('superstore_cleaned.csv', index=False)
print("Done! File saved as superstore_cleaned.csv")

#Saves your clean dataframe as a new CSV file
#index=False prevents pandas from adding an extra numbered column at the start
#The file will be saved in the same folder as your Jupyter Notebook

Done! File saved as superstore_cleaned.csv
